# EOPF GeoZarr Format — Interactive Exploration

This notebook demonstrates the usefulness of the **EOPF Zarr format** for cloud-native Sentinel-2 analysis:

- **No download required** — data streams directly from object storage
- **Analysis-ready** — the `eopf-zarr` xarray backend delivers properly georeferenced, scaled data
- **Resolution-aware** — request any resolution on open; bands are resampled on-the-fly
- **Interactive visualization** — explore imagery and indices with `hvplot`

We'll work over the **Italian Alps** (Aosta Valley region) using Sentinel-2 L2A data.

## 1. Search the STAC catalog

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pystac_client

catalog = pystac_client.Client.open("https://stac.core.eopf.eodc.eu/")

# Aosta Valley, Italian Alps
bbox = [7.2, 44.5, 7.4, 44.7]

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime="2025-04-01/2025-06-01",
    query={"eo:cloud_cover": {"lt": 20}},
)

items = list(search.items())
print(f"Found {len(items)} items")
for item in items:
    print(f"  {item.id}  cloud_cover={item.properties.get('eo:cloud_cover', 'N/A')}%")

## 2. Open with the `eopf-zarr` xarray backend

The `xarray-eopf` package registers an `eopf-zarr` engine that:
- Reads the EOPF Zarr product structure
- Applies scale/offset automatically (returning physical reflectance values)
- Attaches proper CRS coordinates (`x`, `y` in projected metres + `spatial_ref`)
- Resamples all bands to the requested `resolution` on-the-fly

In [ ]:
!pip install xarray-eopf

In [ ]:
import xarray as xr

# Pick the least-cloudy scene
item = sorted(items, key=lambda i: i.properties.get('eo:cloud_cover', 999))[0]
url = item.assets['product'].href
print(f"Opening: {item.id}")
print(f"URL: {url}")

# Open at 60m resolution for fast exploration — change to 10m resolution for full detail
ds = xr.open_dataset(url, engine='eopf-zarr', resolution=60, chunks={})
ds

Notice:
- **`x` / `y` coordinates** in UTM metres — no manual georeferencing needed
- **`spatial_ref`** variable carries the full CRS (EPSG:32632)
- **All bands** at a single consistent resolution via one `open_dataset` call
- Values are **float64 surface reflectance** (scale/offset already applied)

## 3. Clip to our area of interest

Reproject the bbox to UTM and slice — no download of the full 10980×10980 tile.

In [ ]:
import pyproj

transformer = pyproj.Transformer.from_crs("EPSG:4326", "EPSG:32632", always_xy=True)
x_min, y_min = transformer.transform(bbox[0], bbox[1])
x_max, y_max = transformer.transform(bbox[2], bbox[3])
print(f"AOI in UTM: x=[{x_min:.0f}, {x_max:.0f}]  y=[{y_min:.0f}, {y_max:.0f}]")

aoi = ds.sel(x=slice(x_min, x_max), y=slice(y_max, y_min))
aoi

## 4. True-colour RGB composite

In [ ]:
import hvplot.xarray  # noqa: registers hvplot accessor
import numpy as np

# Stack RGB bands and compute into memory (small AOI at 60 m)
rgb = xr.concat(
    [aoi['b04'], aoi['b03'], aoi['b02']],  # R, G, B
    dim='band'
).assign_coords(band=['red', 'green', 'blue']).compute()

# Percentile stretch for display
lo, hi = np.nanpercentile(rgb.values, [2, 98])
rgb_stretched = ((rgb.clip(lo, hi) - lo) / (hi - lo)).clip(0, 1)

rgb_stretched.hvplot.rgb(
    x='x', y='y',
    bands='band',
    crs='EPSG:32632',
    tiles='OSM',
    title=f"True Colour — {item.id[:25]}…",
    frame_width=500,
)

## 5. NDVI — Normalised Difference Vegetation Index

$$\text{NDVI} = \frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red}}$$

Because the data is already in physical reflectance units, NDVI is a one-liner.

In [ ]:
nir = aoi['b08'].compute()
red = aoi['b04'].compute()

ndvi = (nir - red) / (nir + red)
ndvi.name = 'NDVI'

ndvi.hvplot(
    x='x', y='y',
    cmap='RdYlGn',
    clim=(-0.2, 0.8),
    crs='EPSG:32632',
    tiles='OSM',
    title='NDVI — vegetation health',
    frame_width=500,
    colorbar=True,
    alpha=0.7,
)

## 6. NDSI — Snow/Ice Index

$$\text{NDSI} = \frac{\text{Green} - \text{SWIR1}}{\text{Green} + \text{SWIR1}}$$

Useful for alpine scenes — values > 0.4 typically indicate snow/ice.

In [ ]:
green = aoi['b03'].compute()
swir  = aoi['b11'].compute()

ndsi = (green - swir) / (green + swir)
ndsi.name = 'NDSI'

ndsi.hvplot(
    x='x', y='y',
    cmap='Blues_r',
    clim=(-0.5, 1.0),
    crs='EPSG:32632',
    tiles='OSM',
    title='NDSI — snow / ice cover',
    frame_width=500,
    colorbar=True,
    alpha=0.7,
)

## 7. Side-by-side comparison with `+` layout

In [ ]:
true_color = rgb_stretched.hvplot.rgb(
    x='x', y='y',
    bands='band',
    crs='EPSG:32632',
    tiles='OSM',
    title=f"True Colour — {item.id[:25]}…",
    frame_width=380,
)
p_ndsi = ndsi.hvplot(
    x='x', y='y', cmap='Blues_r', clim=(-0.5, 1.0),
    crs='EPSG:32632', tiles='OSM',
    title='NDSI', frame_width=380, colorbar=True, alpha=0.7,
)

true_color + p_ndsi

## 8. Comparing resolutions — zoom in at 10m resolution

One of the key benefits of the EOPF Zarr format is **lazy, resolution-on-demand** access.
Re-open at 10 m for a sharper look — only the AOI pixels are fetched.

In [ ]:
ds_10m = xr.open_dataset(url, engine='eopf-zarr', resolution=10, chunks={})
aoi_10m = ds_10m.sel(x=slice(x_min, x_max), y=slice(y_max, y_min))

nir_10m = aoi_10m['b08'].compute()
red_10m = aoi_10m['b04'].compute()
ndvi_10m = (nir_10m - red_10m) / (nir_10m + red_10m)
ndvi_10m.name = 'NDVI'

p60 = ndvi.hvplot(
    x='x', y='y', cmap='RdYlGn', clim=(-0.2, 0.8),
    rasterize=True, crs='EPSG:32632', tiles='OSM',
    title='NDVI @ 60m resolution', frame_width=380, colorbar=True, alpha=0.7,
)
p10 = ndvi_10m.hvplot(
    x='x', y='y', cmap='RdYlGn', clim=(-0.2, 0.8),
    rasterize=True, crs='EPSG:32632', tiles='OSM',
    title='NDVI @ 10m resolution', frame_width=380, colorbar=True, alpha=0.7,
)

p60 + p10

## 9. Why EOPF Zarr?

| Feature | Traditional SAFE/GeoTIFF | EOPF Zarr |
|---|---|---|
| Format | Zip archive of JP2/TIFF | Cloud-native chunked Zarr |
| Download required? | Yes (1–8 GB) | No — stream only what you need |
| CRS/georef | Manual reprojection | Built-in via `spatial_ref` + coords |
| Scale/offset | Manual application | Auto-applied by backend |
| Multi-resolution | Separate files per resolution | Single open, `resolution=` param |
| xarray integration | Via rioxarray/stackstac | Native `eopf-zarr` engine |
| Parallel/Dask | Requires rechunking | Natively chunked |

Combined with a STAC catalog, this workflow scales from a laptop to a cloud cluster without changing a line of analysis code.